# MCP 入門工作坊 — Hands-on（Colab 版）

> NAPAI · SIGAgent · MCP 工作坊 · Segment 4

**不用在自己電腦裝任何東西。** 每個格子左邊有 ▶，從上到下一個一個按。

跑完你會：
1. 在瀏覽器裡得到一個會自己選工具的 AI 助理（完整 web 介面）
2. 把它改成 **你自己領域** 的助理（換資料、改說明書，不寫一行新邏輯）

⏱ 約 10 分鐘環境 + 30 分鐘 L1 練習

## Section 1 · 環境準備（約 3 分鐘，跑一次就好）

In [2]:
# === 1. 下載教材 + 裝 Node 20 與 uv ===
import os
# 用絕對路徑,重跑這格也不會把目錄越疊越深(避免 .../mini-project/nchu-mcp-workshop-2026/mini-project)
%cd /content
if not os.path.exists('/content/nchu-mcp-workshop-2026'):
    !git clone -q https://github.com/UDICatNCHU/nchu-mcp-workshop-2026
%cd /content/nchu-mcp-workshop-2026/mini-project

# Colab 預設 Node 太舊,裝 Node 20（NodeSource）
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1

# uv：Python 套件管理,MCP 工具伺服器用
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

!node --version
!uv --version

/content
/content/nchu-mcp-workshop-2026/mini-project
v20.20.2
uv 0.11.16 (x86_64-unknown-linux-gnu)


In [3]:
# === 2. 安裝相依套件（約 1 分鐘）===
!cd backend-node && npm install --silent
!cd mcp-server-py && uv sync --quiet
# 讓 notebook 也能直接 import 工具來 demo（L2/L3 的「直接呼叫」cell 用）
!pip install -q mcp httpx
print('✅ 依賴安裝完成')

✅ 依賴安裝完成


In [4]:
# === 3. 設定 Claude API key（講師會發給你）===
# 兩種方式擇一,都不會把 key 留在畫面或檔案裡:
#   (A) Colab Secrets：點左側 🔑 圖示,新增名稱 ANTHROPIC_API_KEY、貼上 value、開啟存取
#       → 設過一次,之後重開 runtime 都免再貼。
#   (B) 沒設 Secret 的話,執行這格會跳出輸入框,把 key 貼進去按 Enter。
import os

key = ''
try:
    from google.colab import userdata
    key = userdata.get('ANTHROPIC_API_KEY') or ''   # 讀 Colab Secrets(🔑)
except Exception:
    key = ''

if not key:
    import getpass
    key = getpass.getpass('貼上 Claude API key（sk-ant-...）後按 Enter: ')

os.environ['ANTHROPIC_API_KEY'] = key.strip()
os.environ['LLM_PROVIDER'] = 'claude'
assert os.environ['ANTHROPIC_API_KEY'].startswith('sk-ant-'), '⚠ key 格式怪怪的,應以 sk-ant- 開頭'
print('✅ key 已設定')

貼上 Claude API key（sk-ant-...）後按 Enter: ··········
✅ key 已設定


In [5]:
import json

config_path = '/content/nchu-mcp-workshop-2026/mini-project/config.json'

with open(config_path) as f:
    config = json.load(f)

config['mcpServers']['arxiv_tool'] = {
    "command": "uv",
    "args": ["--directory", "mcp-server-py", "run", "python", "arxiv_tool.py"]
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print('✅ arxiv_tool 已加入 config.json')

✅ arxiv_tool 已加入 config.json


## Section 2 · 啟動助理

下面這格啟動 backend（它會自動開出 Python MCP 工具子程序）。
**L1 改完檔案後，回來重跑這一格就能套用改動。**

In [6]:
# === 4. 啟動 / 重啟 server ===
import subprocess, time, os

def _kill_port_3000():
    # 殺掉所有佔用 3000 的 node(含重複按 / 遺失 handle 造成的孤兒程序),確保新 server 能 bind。
    try:
        subprocess.run(['pkill', '-9', '-f', 'node server.js'], capture_output=True)
    except Exception:
        pass
    time.sleep(1.5)

def start_server():
    # 重試 2 次:每次都先把舊的殺乾淨,避免 EADDRINUSE。
    for attempt in range(2):
        _kill_port_3000()
        logf = open('/content/server.log', 'w')
        proc = subprocess.Popen(
            ['node', 'server.js'], cwd='backend-node',
            env=os.environ, stdout=logf, stderr=subprocess.STDOUT, text=True,
        )
        log = ''
        ok = False
        for _ in range(25):
            time.sleep(1)
            try: log = open('/content/server.log').read()
            except FileNotFoundError: log = ''
            if 'Mini AI Assistant' in log:
                ok = True
                break
            if 'EADDRINUSE' in log:        # 還有殘留佔用 → 跳出去重試(會再 pkill 一次)
                proc.kill()
                break
        if ok:
            print('✅ server 啟動成功')
            # 從 log 撈出實際註冊的工具(每行一組 server),讓你知道「現在這個助理會什麼」
            tools = [l.strip() for l in log.splitlines() if l.strip().startswith('✓')]
            if tools:
                print(f'\n你的助理現在掛了這些工具（共 {len(tools)} 組,每行一組）——')
                print('上面聊天框可以問跟它們有關的任何問題：')
                for line in tools:
                    print('   ', line)
            return
    print('⚠ server 還沒就緒,最後 log：')
    print(open('/content/server.log').read()[-1200:])

start_server()

✅ server 啟動成功

你的助理現在掛了這些工具（共 6 組,每行一組）——
上面聊天框可以問跟它們有關的任何問題：
    ✓ hello_tool → get_english_center_info
    ✓ teachers_tool → search_teachers, get_teacher_detail
    ✓ weather_tool → get_weather
    ✓ library_tool → search_library
    ✓ course_tool → search_courses
    ✓ arxiv_tool → search_arxiv


## Section 3 · 打開聊天介面

下面那格會把聊天介面**直接顯示在格子下方**。但在你開始問之前 —— 先看看這個助理**現在會什麼**（剛剛啟動那格也印出了它掛了哪些工具）。

它預設掛了 5 組工具，你可以問跟這些有關的**任何問題**（句子隨便講，LLM 會自己抓重點、自己決定要呼叫哪一支）：

| 它會什麼 | 背後工具 | 複製這題問問看 |
|---|---|---|
| 🏫 英文中心資訊 | `hello_tool` | 英文中心幾點開門？ |
| 👥 老師 / 研究領域 | `teachers_tool` | 有誰在做電腦視覺？他們的 email？ |
| 🌤 天氣 | `weather_tool` | 台中今天天氣如何？ |
| 📚 圖書館館藏 | `library_tool` | 圖書館有《原子習慣》這本書嗎？ |
| 📖 3018 門真實課程 | `course_tool` | 有教深度學習的課嗎？老師是誰？ |

> 不知道先問什麼，就從上表挑一題複製貼上。問完想知道「它背後到底做了什麼」→ 跑再下面那格，看它呼叫了哪些工具、跑了幾輪。

In [7]:
# === 5A. 把聊天介面顯示在下方（主要方式）===
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(3000, height=620)

<IPython.core.display.Javascript object>

### 👀 看 loop 在背後做了什麼（本工作坊的核心）

在上面聊天框問一題（例如：「資工系深度學習的課,老師有什麼論文?」），
然後跑下面這格 —— 你會**看到 LLM 自己呼叫了哪些工具、跑了幾輪**：

In [8]:
# === 印出剛剛的 tool_use 軌跡:loop 在背後怎麼跑 ===
log = open('/content/server.log').read()
calls = [l for l in log.splitlines() if '[tool_use]' in l]
if not calls:
    print('還沒看到 tool_use —— 先在上面聊天框問一個需要查資料的問題,再跑這格')
else:
    print('LLM 這幾輪自己呼叫的工具(每行一輪):')
    for c in calls[-12:]:
        print('  →', c.split('[tool_use]')[-1].strip())
    print(f'\n共 {len(calls)} 次 tool_use —— 你沒寫任何流程,全是 LLM 自己決定要叫誰、填什麼')

LLM 這幾輪自己呼叫的工具(每行一輪):
  → search_library

共 1 次 tool_use —— 你沒寫任何流程,全是 LLM 自己決定要叫誰、填什麼


### 萬一上面沒出現介面 / 卡住？

跑下面這格，用 cloudflared 開一個 **公開網址**（新分頁全螢幕，零帳號）。
只在上面那格不動時才需要跑。

In [9]:
# === 5B. 備援：cloudflared 公開網址 ===
import subprocess, time, re, urllib.request

# 0) 先確認本機 server 活著(不然 tunnel 開了也沒東西可服務)
try:
    urllib.request.urlopen('http://localhost:3000', timeout=5)
    print('✅ 本機 server OK')
except Exception as e:
    print('❌ server 沒回應 → 先回去重跑「啟動 server」那格。', e)

!pkill -f cloudflared 2>/dev/null
time.sleep(2)
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://localhost:3000'],
                 stdout=open('/content/tunnel.log','w'),
                 stderr=subprocess.STDOUT, text=True)

# 等到 URL 出現「且」隧道連線真的註冊完成 — 只看 URL 會太早,DNS 解不到
url, registered = None, False
for _ in range(45):
    time.sleep(1)
    try: log = open('/content/tunnel.log').read()
    except FileNotFoundError: log = ''
    if not url:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
        if m: url = m.group(0)
    if 'Registered tunnel connection' in log:
        registered = True
    if url and registered: break

if url and registered:
    print('🌐 在新分頁打開：', url)
    print('⏳ 若顯示「找不到伺服器」,是 DNS 還沒生效 — 等 20-30 秒再重整即可')
else:
    print('⚠ 隧道沒完全建立。建議改用上面的 5A iframe(最穩),或再跑一次這格。')
    print(open('/content/tunnel.log').read()[-800:])

✅ 本機 server OK
^C
🌐 在新分頁打開： https://dozen-champion-honor-reconstruction.trycloudflare.com
⏳ 若顯示「找不到伺服器」,是 DNS 還沒生效 — 等 20-30 秒再重整即可


## 【L1】把它改成你自己領域的助理（課堂主練習）

現在這個助理只懂「英文中心」。接下來 **不寫一行新 Python 邏輯**，
把它改成你的：實驗室介紹 / 課程問答 / 研究成果 / 任何你想被自然語言問的資料。

三步：① 換資料 → ② 改說明書(docstring) → ③ 重啟 + 驗證

In [11]:
# === L1 ① 換成你的資料 ===
# 改成你自己領域的內容,然後按 ▶
import json, pathlib

MY_DATA = {
    "lab_name": "法律 AI 應用課程",
    "pi": {"name": "Jackie Chien", "email": "mjib007@gmail.com", "office": "法律系xxx室"},
    "research_areas": ["AI與法律", "反洗錢合規", "數位法治"],
    "recent_publications": [
        {"year": 2025, "venue": "Vocus", "title": "用Claude Skills打造法學教學代理人"},
    ],
    "openings": {"master": 0, "phd": 0, "prerequisites": "修過法律概論"},
    "weekly_meeting": "每週二 14:00 線上"
}

p = pathlib.Path('mcp-server-py/data/english_center.json')
p.write_text(json.dumps(MY_DATA, ensure_ascii=False, indent=2), encoding='utf-8')
print('✅ 資料已寫入', p)

✅ 資料已寫入 mcp-server-py/data/english_center.json


In [ ]:
# === L1 ② 改 docstring（tool 對 LLM 的「說明書」）===
# LLM 靠這段文字決定「要不要呼叫這個工具」。寫清楚『使用情境』很重要。
TOOL_DOCSTRING = """取得法律 AI 應用課程完整資訊。。

使用情境：使用者詢問研究室的任何資訊,包含研究方向、
PI 聯絡方式、近期論文、招生名額與門檻、週會時間時呼叫。

回傳 JSON 字串。"""

# 把新 docstring 寫回 hello_tool.py（其餘程式不動）
import re, pathlib
path = pathlib.Path('mcp-server-py/hello_tool.py')
src = path.read_text(encoding='utf-8')
new_func = (
    "@mcp.tool()\n"
    "def get_english_center_info() -> str:\n"
    '    """' + TOOL_DOCSTRING + '"""\n'
    "    if not INFO:\n"
    '        return json.dumps({"error": "資料檔不存在"}, ensure_ascii=False)\n'
    "    return json.dumps(INFO, ensure_ascii=False, indent=2)"
)
src = re.sub(r'@mcp\.tool\(\)\ndef get_english_center_info.*?indent=2\)',
             new_func, src, flags=re.DOTALL)
path.write_text(src, encoding='utf-8')
print('✅ docstring 已更新')

✅ docstring 已更新


In [ ]:
# === L1 ③ 重啟 server 讓改動生效,再打開介面 ===
start_server()
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(3000, height=620)
# 試問你自己的資料,例如:「研究室在研究什麼?」「PI 的 email?」「碩士班有名額嗎?」

### ✅ 驗證（做完代表 L1 完成）

- [ ] 問「研究室在研究什麼？」→ 答得出你 JSON 裡的內容
- [ ] 問「今天天氣？」→ AI **不會** 亂呼叫你的工具（它知道不相關）
- [ ] 問一個你資料裡 **沒有** 的東西（例如「有實習嗎？」）→ AI 老實說不知道，**不編造**

最後一項最關鍵 —— tool-calling 正確運作的標誌是「沒有的東西不會胡編」。

## 【L2】有參數的搜尋工具：看 LLM 自己選工具、填參數

L1 的工具沒有參數（讀整包 JSON）。真實工具常常要**帶參數**搜尋。
內建的 `teachers_tool` 有 `search_teachers(keyword, limit)` 跟 `get_teacher_detail(name)`。

**① 在聊天介面**問這幾題，看它怎麼選工具、填參數：
- 「有誰在做電腦視覺？」 → `search_teachers(keyword="電腦視覺")`
- 「有做 CV 的老師嗎？他們的 email？」 → **兩次** tool call：先 search 再 detail（LLM 自己串）

下面兩格讓你**看到程式碼**、並**直接改參數呼叫**，搞懂背後在幹嘛。

In [ ]:
# === L2 ① 偷看工具原始碼：這就是讓它運作的全部程式 ===
print(open('mcp-server-py/teachers_tool.py').read())
# 注意 search_teachers(keyword: str, limit: int) 的 type hint —
# FastMCP 會把它自動變成 LLM 看到的 JSON Schema,LLM 據此決定填什麼參數。

In [ ]:
# === L2 ② 直接呼叫工具(繞過 LLM),改關鍵字再按 ▶ 看輸出 ===
import importlib.util
def _load(name):
    s = importlib.util.spec_from_file_location(name, f'mcp-server-py/{name}.py')
    m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
tt = _load('teachers_tool')
search = getattr(tt.search_teachers, 'fn', tt.search_teachers)  # 取出被 FastMCP 包起來的原函式

print(search('電腦視覺', 3))   # ← 改成你想搜的領域 / 姓名 / 職稱,再按 ▶
# LLM 在聊天裡做的,就是自己決定上面這行的兩個參數。

> **🔑 直接呼叫 vs 透過 agent — 差在哪？**
> - 你剛跑的：`search_teachers("電腦視覺", 3)` ← **你**決定關鍵字、自己呼叫
> - chat 裡問「有誰在做電腦視覺？」 ← **LLM** 自己從句子萃取出 `keyword="電腦視覺"`、自己決定呼叫哪支工具
>
> agent 的價值不是「能呼叫工具」,而是 **LLM 自己判斷該用哪支、填什麼參數、要不要再追問下一步**。直接呼叫只是讓你看清工具本身長怎樣。

## 【L3 · 進階】呼叫外部 API：圖書館館藏查詢

L1/L2 的工具讀**本機資料**。真實世界的工具常常要**打外部 API**。
內建的 `library_tool` 即時查 **中央大學圖書館館藏**（Ex Libris Primo 公開端點，**免 API key**）。

**① 在聊天介面**問：「圖書館有沒有《原子習慣》這本書？」「館藏裡關於機器學習的書？」

下面兩格一樣讓你**看程式碼** + **直接呼叫**，看它真的打了外部 API。

In [ ]:
# === L3 ① 偷看工具原始碼:async + httpx 打外部 API ===
print(open('mcp-server-py/library_tool.py').read())
# 注意它用 httpx.AsyncClient 打 Primo 的 /primaws/rest/pub/pnxs 端點,
# 免 API key,回 JSON 後抽出 title/creator/publisher/year。

In [ ]:
# === L3 ② 直接呼叫工具,改書名/主題再按 ▶ ===
# Colab/Jupyter 本身在跑 event loop,不能用 asyncio.run();在 cell 裡直接 await 即可。
import importlib.util
def _load(name):
    s = importlib.util.spec_from_file_location(name, f'mcp-server-py/{name}.py')
    m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
lt = _load('library_tool')
search_lib = getattr(lt.search_library, 'fn', lt.search_library)  # async 函式

print(await search_lib('原子習慣', 3))   # ← 改成你想查的書名 / 主題,再按 ▶
# 這是真的即時去打中央大學圖書館的 API,不是假資料。

In [ ]:
# === L3 ③ (選做) 換成你自己學校的圖書館 ===
# library_tool 預設查中央大學。其他用 Primo 的學校(含中興)只要改這 4 個 env,
# 重啟 server 後在聊天介面就會查你的學校。找法:到貴校圖書館檢索頁看網址列 vid=。
import os
os.environ['PRIMO_HOST']  = 'ncu.primo.exlibrisgroup.com'   # ← 改成貴校 host
os.environ['PRIMO_VID']   = '886UST_NCU:886UST_NCU'         # ← 改成貴校 vid
os.environ['PRIMO_SCOPE'] = 'MyInstitution'
os.environ['PRIMO_TAB']   = 'nculib'                        # ← 改成貴校 tab
start_server()
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(3000, height=620)

## 【L4】大資料集：在 3018 門真實課程上搜尋

前面的工具資料都不大。真實情境常常是**幾千筆**資料 —— 這支 `course_tool`
查 **中興大學 114-2 學期全部 3018 門課**（真實開課資料，含課程簡述）。

**① 在聊天介面**問問看，最能體會「大資料 + 自然語言」的威力：
- 「有沒有教深度學習的課？」
- 「資工系有哪些選修？」
- 「王銘滄老師開什麼課？」
- 「我想找關於攝影的課」

> 重點：3018 筆**不可能全塞給 LLM**（context 會爆）。工具端先**篩好**再回前幾筆 —— 這就是「工具負責檢索、LLM 負責理解」的分工，也是 Segment 5 要談的 scale 議題的縮影。

In [ ]:
# === L4 ① 偷看工具原始碼:大資料集怎麼搜 ===
print(open('mcp-server-py/course_tool.py').read())
# 注意 search_courses 先在 3018 筆裡篩,只回 limit 筆 —— 不是把整包丟給 LLM。

In [ ]:
# === L4 ② 直接呼叫,改關鍵字/系所再按 ▶ ===
import importlib.util, json
def _load(name):
    s = importlib.util.spec_from_file_location(name, f'mcp-server-py/{name}.py')
    m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
ct = _load('course_tool')
print('資料集共', len(ct.COURSES), '門課')
search_c = getattr(ct.search_courses, 'fn', ct.search_courses)

print(search_c('深度學習', department='', limit=5))   # ← 改關鍵字 / 系所(如 '資工')再按 ▶
# total 是總命中數,results 只回前 limit 筆 —— LLM 看到的就是這份篩過的小清單。

## 【L5 · 挑戰】給 agent 一個它「本來沒有」的能力

這關是整個工作坊的核心體驗 —— 親眼看「**工具 = 給 LLM 它原本辦不到的能力**」。

**① 先問一個它答不出來的**：在上面聊天框問 **「今天星期幾？」**

LLM 沒有「現在」的概念（訓練資料有截止日、它也沒有時鐘）—— 它會說不知道、或亂猜一個日期。**先看它失敗**,再往下做。

In [ ]:
# === ② 給它一支「日期工具」(可改 — 加你想要的計算/邏輯) ===
# 這支用 datetime 拿到『真的今天』—— 正是 LLM 自己辦不到的事。
TOOL_CODE = '''from mcp.server.fastmcp import FastMCP
import datetime, json
mcp = FastMCP("today_tool")

@mcp.tool()
def get_today(until: str = "") -> str:
    """回傳今天的日期與星期幾。
    若帶 until(格式 YYYY-MM-DD),另外算出距離那天還有幾天。
    使用情境:使用者問今天幾號/星期幾、或距離某個日期還有幾天時呼叫。"""
    today = datetime.date.today()
    out = {"今天": today.isoformat(), "星期": "一二三四五六日"[today.weekday()]}
    if until:
        try:
            out[f"距離 {until} 還有(天)"] = (datetime.date.fromisoformat(until) - today).days
        except ValueError:
            out["提示"] = "until 請用 YYYY-MM-DD"
    return json.dumps(out, ensure_ascii=False)

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
import pathlib, json
pathlib.Path('mcp-server-py/today_tool.py').write_text(TOOL_CODE, encoding='utf-8')
cfg = json.load(open('config.json'))
cfg['mcpServers']['today_tool'] = {'command':'uv',
    'args':['--directory','mcp-server-py','run','python','today_tool.py']}
json.dump(cfg, open('config.json','w'), ensure_ascii=False, indent=2)
print('✅ today_tool 已加上 + 註冊,重啟 server 中…')
start_server()
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(3000, height=620)

**③ 再問一次**：「今天星期幾？」、或進階問 **「距離 2026-06-30 還有幾天？」**

這次它答對了 —— 因為它呼叫了你剛給的 `get_today`（進階那題還會自己填 `until="2026-06-30"`）。

> 🎯 **你剛剛做的事 = MCP 的全部意義**：LLM 本來辦不到 → 你給它一支工具 → 它有了新能力。Segment 1 開頭問「怎麼讓 LLM 連上真實世界」,答案就是你剛做的這件事。

## 你剛剛做了什麼？

- 跑起一個 **完整三層** 的 AI 助理：Python MCP 工具 ↔ Node 後端 ↔ web 介面
- 看到了 agentic loop 在背後跑（LLM 自己選工具、填參數、多輪迭代）
- 改資料、試了參數搜尋 / 外部 API / 大資料集,最後**親手寫了一支自己的工具**

### 課後路徑（在自己電腦親手做）
- **L1** 換你的資料 → `docs/labs/L1-customize-your-data.md`（剛在上面做過）
- **L2** 自己寫一支有參數的搜尋工具 → `docs/labs/L2-add-a-search-tool.md`（剛試過內建的 `teachers_tool`）
- **L3** 自己接一個外部 API → `docs/labs/L3-call-external-api.md`（剛試過內建的 `library_tool`）
- **L4** 大資料集搜尋 → 剛試過的 `course_tool`（3018 門課），同樣 pattern 換成你的幾千筆資料
- **完整 web 版**（在自己電腦跑）→ `mini-project/README.md`